In [1]:
import matplotlib.pyplot as plt
import pickle
import numpy as np
import joblib as jl
from tqdm import tqdm
import similaritymeasures as sm
import os
from functions_synthetic_data_generation import *
from functions_synthetic_data_analysis import *
from functions_hmm import *
import sys

plt.style.use('../paper.mplstyle')

print(plt.get_backend())
%matplotlib qt5
print(plt.get_backend())

module://matplotlib_inline.backend_inline
qt5agg


# Simulations generations

## Parameters

In [2]:
from used_parameters_n_20 import *

# Plot HMM fitting result

# Drift Estimation

## Generating "theoretical" average probability sequences

In [3]:
DRIFT_RANGE = np.linspace(0.001,0.11,200)

generate_theoretical_sequences = False # PARAM

if generate_theoretical_sequences:

    temp_args_dict = copy.deepcopy(DEFAULT_ARGS_DICT)
        
    args = [temp_args_dict] + [5000] + [P_CW_INIT_RANGE]
    DRIFT_RANGE = np.linspace(0.001,0.11,50)

    test_average_probability_sequences_list = generate_test_average_probability_sequences_identical_drifts_random_init(DRIFT_RANGE, args)

    with open(f'{MAIN_FOLDER_PATH}/test_average_probability_sequences_1.pkl', 'wb') as file:
        pickle.dump([DRIFT_RANGE, test_average_probability_sequences_list], file)

else:
   
    with open(f'{MAIN_FOLDER_PATH}/test_average_probability_sequences_1.pkl', 'rb') as file:
        DRIFT_RANGE, test_average_probability_sequences_list = pickle.load(file)
test_average_probability_sequences_list = [test_seq[:STEPS_NUMBER] for test_seq in test_average_probability_sequences_list]
print(len(test_average_probability_sequences_list[0]))

40


## Minimizing mean square error

In [4]:
def estimate_drift(simulations_set, model, test_average_probability_sequences_list, drift_range):

    choices_sequences_list = [synth_data['choices'] for synth_data in simulations_set]

    average_proba_sequence_hmm = compute_reconstructed_average_proba_sequence(choices_sequences_list, model)

    mse_list = []

    for test_average_probability_sequence in test_average_probability_sequences_list:
        
        mse_list.append(compute_mean_square_error_v2(average_proba_sequence_hmm, test_average_probability_sequence))

    min_mse = np.min(mse_list)
    min_mse_index = np.where(mse_list==min_mse)[0]
    recovered_drift = drift_range[min_mse_index]

    return recovered_drift, min_mse_index

In [5]:
stacked_recovered_drift_per_states_number = []

for n in range(len(DRIFT_VALUES_ARR)):

    recovered_drift_per_states_number = []
    
    for n_states_index in range(len(STATES_NUMBER_RANGE)):

        recovered_drift_per_states_number.append([])

        for i in range(N_SETS):

            average_proba_sequence_hmm = []

            ##############################
            ### Loading Data and Model ###
            ##############################

            with open(f'{MAIN_FOLDER_PATH}/drift_{n}/simulation_set_{i}.pkl', 'rb') as file:
                simulations_set = pickle.load(file)

            with open(f'{MAIN_FOLDER_PATH}/drift_{n}/hmm_fit_output_{i}.pkl', 'rb') as file:
                fit_output = pickle.load(file)

            # model = fit_output['models'][np.argmax(fit_output['scores'])]
            model = fit_output['models'][n_states_index]

            ####################
            ### Computations ###
            ####################

            test_data = [simulation['choices'] for simulation in simulations_set]
            recovered_drift, _ = estimate_drift(simulations_set, model, test_average_probability_sequences_list, DRIFT_RANGE)
            
            recovered_drift_per_states_number[-1].append(recovered_drift)
            
    stacked_recovered_drift_per_states_number.append(recovered_drift_per_states_number)

# with open(f'{MAIN_FOLDER_PATH}/estimated_drift_list.pkl', 'wb') as file:
#     pickle.dump(stacked_recovered_drift_per_states_number,file)

In [6]:
example_drift = 2

fig=plt.figure(figsize=(4, 2.5), dpi=300, constrained_layout=False, facecolor='w')
gs = fig.add_gridspec(1, 1)
row = gs[:].subgridspec(1, 1, hspace=0.5)

ax1 = plt.subplot(row[0,0])



for i in range(len(STATES_NUMBER_RANGE)):

    temp_recovered_drift_list = stacked_recovered_drift_per_states_number[example_drift][i]
    ax1.scatter(np.ones(N_SETS)*STATES_NUMBER_RANGE[i], temp_recovered_drift_list, marker='+', alpha=0.2, s=8)
    mean = np.mean(temp_recovered_drift_list)
    std = np.std(temp_recovered_drift_list)
    ax1.errorbar(STATES_NUMBER_RANGE[i],mean,std, marker='o', markersize=2, capsize=5)

ax1.axhline(DRIFT_VALUES_ARR[example_drift], color='k', linestyle='--')

# ax1.set_xticks([])
ax1.set_title(f'{N_SETS} sets of {SIMULATIONS_SET_SIZE} simulations')

ax1.set_ylim([0,0.15])
ax1.set_xlabel('Number of states in HMM')
ax1.set_ylabel('Recovered drift')

ax1.legend()

/tmp/ipykernel_862542/3341282898.py:28: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax1.legend()
